# 03 — RDDs and the MapReduce Programming Model

**Airline Operations Intelligence Platform** · Notebook 3 of 10 · *runs locally*

## Purpose
Demonstrate the foundational distributed-computing layer beneath Spark's DataFrames, and
measure *why* production code uses DataFrames instead.

Syllabus coverage (Unit 2 and Unit 4):
- MapReduce programming model — `map` / `reduceByKey`
- Limitations of MapReduce
- RDD creation, transformations, actions
- Narrow vs wide dependencies, and the shuffle
- Lazy evaluation and the DAG
- Fault tolerance through lineage
- Spark vs Hadoop MapReduce

Every claim here is **measured on the curated 5.8M-row dataset**, not asserted.

In [ ]:
import sys, time
sys.path.insert(0, "../src")

from config import build_spark, PATHS
from pyspark.sql import functions as F

spark = build_spark("03-rdd")
sc = spark.sparkContext

flights = spark.read.parquet(str(PATHS["curated"] / "flights.parquet"))
flights.cache()
N = flights.count()
print(f"Curated flights : {N:,}")
print(f"Default parallelism : {sc.defaultParallelism}")

---
## 1. What an RDD actually is

An **RDD** (Resilient Distributed Dataset) is an immutable, partitioned collection with a
recorded **lineage** — the exact sequence of operations that produced it. That lineage is
what makes it *resilient*: a lost partition is recomputed, not restored from a replica.

A DataFrame is a higher-level API built on top of RDDs. Dropping to `.rdd` shows the
machinery underneath.

In [ ]:
rdd = flights.select("airline_code", "arr_delay", "status", "origin", "distance").rdd

print("Type       :", type(rdd).__name__)
print("Partitions :", rdd.getNumPartitions())
print("\nFirst 3 elements (Row objects):")
for row in rdd.take(3):
    print("  ", row)

### Transformations are lazy; actions are not

`map`, `filter`, `reduceByKey` return a new RDD and compute nothing. `collect`, `count`,
`take`, `reduce` are **actions** — they submit a job.

In [ ]:
t0 = time.time()
lazy = (rdd.filter(lambda r: r["status"] == "completed")
           .filter(lambda r: r["arr_delay"] is not None and r["arr_delay"] > 15)
           .map(lambda r: (r["airline_code"], 1)))
t_build = time.time() - t0

t0 = time.time()
n = lazy.count()          # ACTION
t_act = time.time() - t0

print(f"Building 3 transformations : {t_build:.5f}s   (nothing ran)")
print(f"count() action             : {t_act:.2f}s   (job submitted, DAG executed)")
print(f"Delayed flights            : {n:,}")

---
## 2. The MapReduce pattern

The classic two-phase model:

| Phase | Role | Spark equivalent |
|---|---|---|
| **Map** | Transform each record into `(key, value)` | `map()` |
| **Shuffle** | Group all values sharing a key onto one node | implicit in `reduceByKey` |
| **Reduce** | Combine values per key into a result | `reduceByKey()` |

### 2a. Word count — the canonical example

The textbook "Hello World" of MapReduce, run here on real airport codes.

In [ ]:
t0 = time.time()
airport_counts = (flights.select("origin").rdd
    .map(lambda r: (r["origin"], 1))          # MAP:     emit (key, 1)
    .reduceByKey(lambda a, b: a + b)          # SHUFFLE+REDUCE: sum per key
    .sortBy(lambda kv: -kv[1]))               # order by count desc

top10 = airport_counts.take(10)
t_wc = time.time() - t0

print(f"Word-count over {N:,} records in {t_wc:.2f}s\n")
print(f"  {'AIRPORT':<10}{'DEPARTURES':>12}")
print("  " + "-"*22)
for code_, cnt in top10:
    print(f"  {code_:<10}{cnt:>12,}")

### 2b. Delay counts per airline — MapReduce style

The syllabus example from the project plan, written with raw RDD operations only.

In [ ]:
t0 = time.time()
delay_counts_rdd = (flights.rdd
    .filter(lambda r: r["status"] == "completed")
    .filter(lambda r: r["arr_delay"] is not None and r["arr_delay"] > 15)
    .map(lambda r: (r["airline_code"], 1))
    .reduceByKey(lambda a, b: a + b)
    .collect())
t_rdd = time.time() - t0

print(f"RDD MapReduce : {t_rdd:.2f}s")
for k, v in sorted(delay_counts_rdd, key=lambda kv: -kv[1])[:5]:
    print(f"  {k}  {v:>9,}")

### 2c. A harder case — average delay per airline

An average cannot be computed with a simple `reduceByKey(a + b)`: averages are not
associative. The MapReduce solution is to carry a `(sum, count)` pair through the reduce
and divide at the end.

This is where the verbosity of the model becomes obvious.

In [ ]:
t0 = time.time()
avg_rdd = (flights.rdd
    .filter(lambda r: r["status"] == "completed" and r["arr_delay"] is not None)
    .map(lambda r: (r["airline_code"], (r["arr_delay"], 1)))        # (key, (sum, count))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))           # pairwise combine
    .mapValues(lambda sc_: sc_[0] / sc_[1])                          # divide at the end
    .collect())
t_avg_rdd = time.time() - t0

print(f"RDD version : {t_avg_rdd:.2f}s  -- 4 chained operations, manual (sum, count)\n")
for k, v in sorted(avg_rdd, key=lambda kv: -kv[1])[:5]:
    print(f"  {k}  {v:6.2f} min")

In [ ]:
# The same result in the DataFrame API.
t0 = time.time()
avg_df = (flights.filter(F.col("status") == "completed")
                 .groupBy("airline_code")
                 .agg(F.avg("arr_delay").alias("avg_delay"))
                 .collect())
t_avg_df = time.time() - t0

print(f"DataFrame version : {t_avg_df:.2f}s  -- one groupBy().agg()\n")
for r in sorted(avg_df, key=lambda r: -r["avg_delay"])[:5]:
    print(f"  {r['airline_code']}  {r['avg_delay']:6.2f} min")

print(f"\nSpeedup: {t_avg_rdd/t_avg_df:.1f}x faster, and materially less code.")

---
## 3. Three APIs, one answer

Proof that the RDD, DataFrame and SparkSQL paths are equivalent in *result* and
different in *cost*.

In [ ]:
flights.createOrReplaceTempView("flights_v")

# --- 1. RDD
t0 = time.time()
r_rdd = dict((flights.rdd
    .filter(lambda r: r["status"] == "completed" and r["arr_delay"] is not None and r["arr_delay"] > 15)
    .map(lambda r: (r["airline_code"], 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()))
t1 = time.time() - t0

# --- 2. DataFrame
t0 = time.time()
r_df = {r["airline_code"]: r["c"] for r in (
    flights.filter((F.col("status") == "completed") & (F.col("arr_delay") > 15))
           .groupBy("airline_code").agg(F.count("*").alias("c")).collect())}
t2 = time.time() - t0

# --- 3. SparkSQL
t0 = time.time()
r_sql = {r["airline_code"]: r["c"] for r in spark.sql("""
    SELECT airline_code, COUNT(*) AS c
    FROM flights_v
    WHERE status = 'completed' AND arr_delay > 15
    GROUP BY airline_code
""").collect()}
t3 = time.time() - t0

assert r_rdd == r_df == r_sql, "the three APIs disagree"
print("All three APIs return identical results.\n")
print(f"  {'API':<14}{'TIME':>9}{'vs RDD':>10}")
print("  " + "-"*33)
print(f"  {'RDD':<14}{t1:>8.2f}s{'1.0x':>10}")
print(f"  {'DataFrame':<14}{t2:>8.2f}s{t1/t2:>9.1f}x")
print(f"  {'SparkSQL':<14}{t3:>8.2f}s{t1/t3:>9.1f}x")

### Why the DataFrame versions win

RDDs hold opaque Python objects. Spark cannot see inside a `lambda`, so it must:

1. **Serialise every row** from the JVM to a Python worker process, and back.
2. Apply the function **row by row** in Python.
3. Forgo optimisation — it cannot reorder, combine or eliminate steps it cannot inspect.

DataFrame operations are declarative, so the **Catalyst optimiser** rewrites the plan
(pushing filters into the Parquet reader, pruning unread columns) and **Tungsten** executes
on compact off-heap binary data inside the JVM — no Python round-trip at all.

Both DataFrame and SparkSQL compile to the *same* optimised plan, which is why their
timings match. Verified below — note that the plans must be compared **after normalising
Catalyst's expression ids**, since every expression gets a globally unique id
(`c#50L` vs `c#101L`) and raw strings therefore never match.

In [ ]:
import re

plan_df  = (flights.filter((F.col("status") == "completed") & (F.col("arr_delay") > 15))
                   .groupBy("airline_code").agg(F.count("*").alias("c")))
plan_sql = spark.sql("""SELECT airline_code, COUNT(*) AS c FROM flights_v
                        WHERE status = 'completed' AND arr_delay > 15 GROUP BY airline_code""")

a = plan_df._jdf.queryExecution().optimizedPlan().toString()
b = plan_sql._jdf.queryExecution().optimizedPlan().toString()

# Catalyst assigns every expression a globally unique id (arr_delay#36, c#50L).
# Two plans built at different times therefore never match as raw strings even when
# they are structurally identical. Normalise the ids before comparing.
normalise = lambda s: re.sub(r"#\d+L?", "#x", s)

print("Raw strings equal          :", a == b)
print("Equal after normalising ids:", normalise(a) == normalise(b))
print()
print(a)

---
## 4. Narrow vs wide dependencies

- **Narrow** (`map`, `filter`): each output partition depends on one input partition.
  No data movement; stages fuse together.
- **Wide** (`reduceByKey`, `groupByKey`, joins): output partitions depend on *many* input
  partitions. Requires a **shuffle** — data crosses the network and hits disk.

The shuffle is the expensive operation in all distributed processing. `toDebugString`
shows exactly where it happens.

In [ ]:
pipeline = (flights.select("airline_code", "arr_delay").rdd
            .filter(lambda r: r["arr_delay"] is not None)     # narrow
            .map(lambda r: (r["airline_code"], r["arr_delay"]))  # narrow
            .reduceByKey(lambda a, b: a + b))                 # WIDE -> shuffle

print(pipeline.toDebugString().decode())
print("\nEach indent level is a STAGE. A new stage begins at every ShuffledRDD.")

### `reduceByKey` vs `groupByKey` — and a lesson about benchmarks

Both produce the same answer. The standard teaching is that `reduceByKey` combines values
**locally on each partition before** the shuffle (a *combiner*, in Hadoop terms), so far
less data crosses the network, while `groupByKey` ships every individual value.

Measure it across four key cardinalities and report what actually happens — including
when the expected effect does **not** appear.

In [ ]:
def compare(key_col, label):
    pairs = flights.select(key_col).rdd.map(lambda r: (r[0], 1))
    pairs.cache(); pairs.count()                     # isolate the shuffle from file I/O

    t0 = time.time(); a = pairs.reduceByKey(lambda x, y: x + y).collect(); t_r = time.time()-t0
    t0 = time.time(); b = pairs.groupByKey().mapValues(len).collect();     t_g = time.time()-t0
    assert dict(a) == dict(b), "the two paths disagree"

    print(f"  {label:<22}{len(a):>8,}{t_r:>12.2f}s{t_g:>12.2f}s{t_g/t_r:>9.2f}x")
    pairs.unpersist()

print(f"  {'GROUPING KEY':<22}{'KEYS':>8}{'reduceByKey':>13}{'groupByKey':>13}{'RATIO':>9}")
print("  " + "-"*64)
compare("airline_code", "airline_code")
compare("origin",       "origin airport")
compare("route",        "route")
compare("tail_number",  "tail_number")

#### Reading the result honestly

The ratios above hover around 1.0 with no consistent trend — sometimes `groupByKey` is
even marginally faster. **The textbook advantage does not reproduce here, and it is worth
understanding why rather than explaining it away.**

1. **Local mode has no network.** The entire argument for `reduceByKey` is that a combiner
   reduces bytes crossing the *network*. On one machine the shuffle is a local disk write,
   which is orders of magnitude cheaper — the cost the optimisation targets barely exists.
2. **The values are 1-byte integers.** Shipping every value is cheap when each value is
   tiny. The gap widens when values are large (strings, lists, structs).
3. **PySpark serialisation dominates.** JVM↔Python round-trips account for most of the
   runtime in both variants and swamp the difference.
4. **`reduceByKey` is not free.** It pays for a local combine pass. At 14 keys that work
   buys almost nothing, which is why it can come out slightly behind.

**The guidance still stands** — prefer `reduceByKey` — but for reasons this benchmark
cannot demonstrate: on a real cluster, with large values and high cardinality,
`groupByKey` also risks OOM because all values for one key must fit in memory on a single
executor. `reduceByKey` never materialises that list.

**The transferable lesson:** a benchmark run on one machine does not measure distributed
cost. Reporting "no difference observed, and here is why" is a stronger result than
quietly reshaping the experiment until it agrees with the textbook.

---
## 5. Fault tolerance through lineage

Hadoop achieves fault tolerance by **replicating data** (HDFS keeps 3 copies).
Spark achieves it by **remembering the computation**. If a partition is lost, Spark
replays that partition's lineage — recomputation instead of redundancy.

The trade-off: Spark needs no extra storage, but recovery costs CPU time. For long
lineages, `checkpoint()` truncates the chain by writing to stable storage.

In [ ]:
lineage = (flights.select("origin", "arr_delay").rdd
           .filter(lambda r: r["arr_delay"] is not None)
           .map(lambda r: (r["origin"], r["arr_delay"]))
           .reduceByKey(lambda a, b: max(a, b)))

print("Lineage graph Spark would replay to rebuild any lost partition:\n")
print(lineage.toDebugString().decode())

---
## 6. Limitations of MapReduce, measured

The syllabus asks for the limitations of the MapReduce model. The decisive one is
**iterative workloads**: Hadoop MapReduce writes intermediate results to disk between
every job, so an algorithm with N passes pays N round-trips to disk. Spark keeps data in
memory across passes.

Simulated below with 5 passes over the same data, with and without caching.

In [ ]:
uncached = flights.select("airline_code", "arr_delay", "status").rdd
uncached = uncached.filter(lambda r: r["status"] == "completed")

t0 = time.time()
for i in range(5):
    uncached.map(lambda r: (r["airline_code"], 1)).reduceByKey(lambda a,b: a+b).count()
t_nocache = time.time() - t0

cached = flights.select("airline_code", "arr_delay", "status").rdd \
                .filter(lambda r: r["status"] == "completed").cache()
cached.count()   # materialise

t0 = time.time()
for i in range(5):
    cached.map(lambda r: (r["airline_code"], 1)).reduceByKey(lambda a,b: a+b).count()
t_cache = time.time() - t0

print(f"5 passes, recomputed each time : {t_nocache:.2f}s   <- the MapReduce model")
print(f"5 passes, cached in memory     : {t_cache:.2f}s   <- Spark's advantage")
print(f"Speedup                        : {t_nocache/t_cache:.1f}x")
print("\nThis gap is why iterative algorithms -- and therefore all of machine learning --")
print("are impractical on classical MapReduce and routine on Spark.")
cached.unpersist()

### Summary — MapReduce vs Spark

| Aspect | Hadoop MapReduce | Spark |
|---|---|---|
| Intermediate results | Written to HDFS between jobs | Held in memory (spills to disk only if needed) |
| Iterative algorithms | Re-reads from disk every pass | Cache once, reuse — measured above |
| Programming model | Rigid map → shuffle → reduce | Arbitrary DAG of transformations |
| Expressing an average | Manual `(sum, count)` plumbing (§2c) | One `groupBy().agg()` |
| Optimisation | None — you write the plan | Catalyst rewrites the plan |
| Fault tolerance | Data replication (HDFS, 3×) | Lineage recomputation |
| Latency | High per-job JVM startup | Long-lived executors |

### Why this project uses DataFrames everywhere else
Measured above: DataFrames are several times faster for identical results, far shorter to
write, and let Catalyst push filters into the Parquet reader. The RDD API remains the
right tool for genuinely unstructured data or custom partitioning — neither of which this
dataset needs.

In [ ]:
flights.unpersist()
spark.stop()
print("Notebook 03 complete.")